In [2]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv("01_raw_crm_input_15000.csv")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (15000, 14)

Columns:
['interaction_id', 'crm_note_id', 'interaction_date', 'rep_id', 'hcp_id', 'city', 'region', 'territory_id', 'hcp_specialization', 'therapeutic_area', 'drug_id', 'drug_name', 'brand_name', 'crm_note']


In [3]:
print("Missing CRM notes:", df["crm_note"].isna().sum())
print("Duplicate CRM notes:", df["crm_note"].duplicated().sum())
print("Empty CRM notes:", (df["crm_note"].fillna("").str.strip() == "").sum())


Missing CRM notes: 0
Duplicate CRM notes: 0
Empty CRM notes: 0


In [4]:
df[["crm_note"]].head(10)

,crm_note
0,Saw HCP re Arthrelis. 2 patients mentioned inj...
1,Met on Arthrelis. HCP reports fewer tolerance ...
2,Met on Arthrelis. the previous adherence conce...
3,The HCP reviewed recent experience with Arthre...
4,Brief discussion focused on Rheumora. payer ap...
5,Quick f/u on Arthrelis. office reports fewer p...
6,Arthrelis discussion. use in routine practice ...
7,Saw HCP re Arthrelis. HCP remains cautious abo...
8,Rheumora discussion. recent approvals have gon...
9,Saw HCP re Renovia. practice wants a simpler d...


In [5]:
# Display complete first 10 CRM notes

for i, note in enumerate(df["crm_note"].head(10), start=1):
    print(f"\n{'='*80}")
    print(f"CRM NOTE {i}")
    print(f"{'='*80}")
    print(note)


CRM NOTE 1
Saw HCP re Arthrelis. 2 patients mentioned injection-site reaction. recent outcomes have been favorable. Nothing else urgent.

CRM NOTE 2
Met on Arthrelis. HCP reports fewer tolerance complaints recently. several patients are not following the regimen consistently. Send patient-support material by Mar 06; owner REP0002.

CRM NOTE 3
Met on Arthrelis. the previous adherence concern has eased. HCP wants more confidence in expected benefit. authorization turnaround remains slow. HCP wants access / pa resource; REP0002 to f/u.

CRM NOTE 4
The HCP reviewed recent experience with Arthrelis during the scheduled visit. HCP is more comfortable with efficacy after seeing recent outcomes. Copay burden is hurting continuation. The HCP asked for affordability resource; follow-up ownership remains with REP0002.

CRM NOTE 5
Brief discussion focused on Rheumora. payer approval process is still too cumbersome.

CRM NOTE 6
Quick f/u on Arthrelis. office reports fewer payer problems this month

In [17]:
# Create temporary word count for EDA

df["word_count_raw"] = df["crm_note"].str.split().str.len()

print("Average words :", round(df["word_count_raw"].mean(), 2))
print("Median words  :", df["word_count_raw"].median())
print("Minimum words :", df["word_count_raw"].min())
print("Maximum words :", df["word_count_raw"].max())

Average words : 25.69
Median words  : 24.0
Minimum words : 2
Maximum words : 73


In [18]:
df["word_count_raw"].describe()

,word_count_raw
count,15000.000000
mean,25.689400
std,11.736086
min,2.000000
25%,17.000000
50%,24.000000
75%,33.000000
max,73.000000


In [19]:
#Create a working copy
df_clean = df.copy()

In [20]:
#Create text_raw
df_clean["text_raw"] = df_clean["crm_note"]

In [21]:
#Lowercase
df_clean["text_lower"] = df_clean["text_raw"].str.lower()

In [22]:
df_clean[["text_raw", "text_lower"]].head(10)

,text_raw,text_lower
0,Saw HCP re Arthrelis. 2 patients mentioned inj...,saw hcp re arthrelis. 2 patients mentioned inj...
1,Met on Arthrelis. HCP reports fewer tolerance ...,met on arthrelis. hcp reports fewer tolerance ...
2,Met on Arthrelis. the previous adherence conce...,met on arthrelis. the previous adherence conce...
3,The HCP reviewed recent experience with Arthre...,the hcp reviewed recent experience with arthre...
4,Brief discussion focused on Rheumora. payer ap...,brief discussion focused on rheumora. payer ap...
5,Quick f/u on Arthrelis. office reports fewer p...,quick f/u on arthrelis. office reports fewer p...
6,Arthrelis discussion. use in routine practice ...,arthrelis discussion. use in routine practice ...
7,Saw HCP re Arthrelis. HCP remains cautious abo...,saw hcp re arthrelis. hcp remains cautious abo...
8,Rheumora discussion. recent approvals have gon...,rheumora discussion. recent approvals have gon...
9,Saw HCP re Renovia. practice wants a simpler d...,saw hcp re renovia. practice wants a simpler d...


In [23]:
#Normalize important abbreviations
import re

def normalize_abbreviations(text):
    # f/u = follow-up
    text = re.sub(r'\bf\s*/\s*u\b', 'follow up', text)

    # PA = prior authorization
    text = re.sub(r'\bpa\b', 'prior authorization', text)

    # "re" used in CRM notes = regarding
    text = re.sub(r'\bre\b', 'regarding', text)

    return text

df_clean["text_normalized"] = (
    df_clean["text_lower"]
    .apply(normalize_abbreviations)
)

In [24]:
df_clean[
    ["text_lower", "text_normalized"]
].head(10)

,text_lower,text_normalized
0,saw hcp re arthrelis. 2 patients mentioned inj...,saw hcp regarding arthrelis. 2 patients mentio...
1,met on arthrelis. hcp reports fewer tolerance ...,met on arthrelis. hcp reports fewer tolerance ...
2,met on arthrelis. the previous adherence conce...,met on arthrelis. the previous adherence conce...
3,the hcp reviewed recent experience with arthre...,the hcp reviewed recent experience with arthre...
4,brief discussion focused on rheumora. payer ap...,brief discussion focused on rheumora. payer ap...
5,quick f/u on arthrelis. office reports fewer p...,quick follow up on arthrelis. office reports f...
6,arthrelis discussion. use in routine practice ...,arthrelis discussion. use in routine practice ...
7,saw hcp re arthrelis. hcp remains cautious abo...,saw hcp regarding arthrelis. hcp remains cauti...
8,rheumora discussion. recent approvals have gon...,rheumora discussion. recent approvals have gon...
9,saw hcp re renovia. practice wants a simpler d...,saw hcp regarding renovia. practice wants a si...


In [25]:
#Remove administrative IDs and dates
def remove_admin_noise(text):
    # Remove representative IDs such as REP0002
    text = re.sub(r'\brep\d+\b', ' ', text, flags=re.IGNORECASE)

    # Remove dates such as "Mar 06", "Jun 11", etc.
    text = re.sub(
        r'\b(?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\s+\d{1,2}\b',
        ' ',
        text,
        flags=re.IGNORECASE
    )

    # Remove standalone numbers
    text = re.sub(r'\b\d+\b', ' ', text)

    # Clean extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

df_clean["text_no_admin"] = (
    df_clean["text_normalized"]
    .apply(remove_admin_noise)
)

In [26]:
df_clean[
    ["text_normalized", "text_no_admin"]
].head(10)

,text_normalized,text_no_admin
0,saw hcp regarding arthrelis. 2 patients mentio...,saw hcp regarding arthrelis. patients mentione...
1,met on arthrelis. hcp reports fewer tolerance ...,met on arthrelis. hcp reports fewer tolerance ...
2,met on arthrelis. the previous adherence conce...,met on arthrelis. the previous adherence conce...
3,the hcp reviewed recent experience with arthre...,the hcp reviewed recent experience with arthre...
4,brief discussion focused on rheumora. payer ap...,brief discussion focused on rheumora. payer ap...
5,quick follow up on arthrelis. office reports f...,quick follow up on arthrelis. office reports f...
6,arthrelis discussion. use in routine practice ...,arthrelis discussion. use in routine practice ...
7,saw hcp regarding arthrelis. hcp remains cauti...,saw hcp regarding arthrelis. hcp remains cauti...
8,rheumora discussion. recent approvals have gon...,rheumora discussion. recent approvals have gon...
9,saw hcp regarding renovia. practice wants a si...,saw hcp regarding renovia. practice wants a si...


In [27]:
#Remove punctuation
def remove_punctuation(text):
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_clean["text_no_punct"] = (
    df_clean["text_no_admin"].apply(remove_punctuation)
)

In [28]:
df_clean[["text_no_admin", "text_no_punct"]].head(10)

,text_no_admin,text_no_punct
0,saw hcp regarding arthrelis. patients mentione...,saw hcp regarding arthrelis patients mentioned...
1,met on arthrelis. hcp reports fewer tolerance ...,met on arthrelis hcp reports fewer tolerance c...
2,met on arthrelis. the previous adherence conce...,met on arthrelis the previous adherence concer...
3,the hcp reviewed recent experience with arthre...,the hcp reviewed recent experience with arthre...
4,brief discussion focused on rheumora. payer ap...,brief discussion focused on rheumora payer app...
5,quick follow up on arthrelis. office reports f...,quick follow up on arthrelis office reports fe...
6,arthrelis discussion. use in routine practice ...,arthrelis discussion use in routine practice f...
7,saw hcp regarding arthrelis. hcp remains cauti...,saw hcp regarding arthrelis hcp remains cautio...
8,rheumora discussion. recent approvals have gon...,rheumora discussion recent approvals have gone...
9,saw hcp regarding renovia. practice wants a si...,saw hcp regarding renovia practice wants a sim...


In [29]:
#Stopword handling
import nltk
nltk.download("stopwords")

from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [30]:
stop_words = set(stopwords.words("english"))

# Keep important negation words
words_to_keep = {
    "no", "not", "nor", "never",
    "neither", "without", "against"
}

stop_words = stop_words - words_to_keep

print("Number of stopwords:", len(stop_words))

Number of stopwords: 194


In [6]:
#important terms
from collections import Counter
import re
import pandas as pd

# Store all words from all CRM notes
all_words = []

# Store unique words for each individual CRM record
document_words = []

for text in df["crm_note"].fillna("").astype(str):

    words = re.findall(r"\b[a-zA-Z]+\b", text.lower())

    all_words.extend(words)
    document_words.append(set(words))

In [7]:
#Calculate occurrence
word_frequency = Counter(all_words)

print("Top 30 words by total occurrence:")
print(word_frequency.most_common(30))

Top 30 words by total occurrence:
[('the', 23785), ('hcp', 15372), ('is', 9850), ('recent', 7075), ('with', 6726), ('to', 6367), ('on', 5973), ('for', 5419), ('patients', 5335), ('in', 5315), ('experience', 5149), ('a', 4623), ('up', 4549), ('has', 4478), ('by', 4360), ('and', 4050), ('current', 3925), ('visit', 3805), ('follow', 3582), ('during', 3533), ('of', 3525), ('patient', 3237), ('been', 3214), ('practice', 3121), ('discussion', 2908), ('evidence', 2772), ('s', 2601), ('clinical', 2598), ('use', 2267), ('remains', 2239)]


In [8]:
#Calculate uniqueness / document frequency
document_frequency = Counter()

for words in document_words:
    for word in words:
        document_frequency[word] += 1

In [9]:
print("Top 30 words by unique documents:")

print(
    document_frequency.most_common(30)
)

Top 30 words by unique documents:
[('hcp', 10478), ('the', 10451), ('is', 7483), ('recent', 6082), ('on', 5569), ('with', 5418), ('to', 5256), ('experience', 5022), ('patients', 4689), ('in', 4665), ('for', 4574), ('by', 4269), ('has', 4059), ('a', 4045), ('up', 4025), ('current', 3642), ('visit', 3625), ('and', 3382), ('during', 3359), ('follow', 3300), ('of', 3259), ('been', 3045), ('patient', 2940), ('practice', 2912), ('discussion', 2838), ('evidence', 2624), ('s', 2557), ('clinical', 2400), ('use', 2172), ('practical', 2125)]


In [10]:
#Create one table containing both
word_stats = pd.DataFrame({
    "word": list(word_frequency.keys()),
    "total_occurrences": [
        word_frequency[word]
        for word in word_frequency.keys()
    ],
    "unique_documents": [
        document_frequency[word]
        for word in word_frequency.keys()
    ]
})

In [11]:
word_stats["document_coverage_pct"] = (
    word_stats["unique_documents"] / len(df) * 100
)

In [12]:
#Sort by occurrence:
word_stats = word_stats.sort_values(
    "total_occurrences",
    ascending=False
)

display(word_stats.head(50))

,word,total_occurrences,unique_documents,document_coverage_pct
28,the,23785,10451,69.673333
1,hcp,15372,10478,69.853333
65,is,9850,7483,49.886667
9,recent,7075,6082,40.546667
61,with,6726,5418,36.120000
56,to,6367,5256,35.040000
18,on,5973,5569,37.126667
75,for,5419,4574,30.493333
4,patients,5335,4689,31.260000
46,in,5315,4665,31.100000


In [34]:
#Remove normal English stopwords for this analysis
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

word_stats = word_stats[
    ~word_stats["word"].isin(ENGLISH_STOP_WORDS)
]

In [35]:
display(
    word_stats.head(50)
)

,word,total_occurrences,unique_documents,document_coverage_pct
1,hcp,15372,10478,69.853333
9,recent,7075,6082,40.546667
4,patients,5335,4689,31.260000
60,experience,5149,5022,33.480000
192,current,3925,3642,24.280000
64,visit,3805,3625,24.166667
77,follow,3582,3300,22.000000
32,patient,3237,2940,19.600000
95,practice,3121,2912,19.413333
81,discussion,2908,2838,18.920000


In [39]:
#Create a candidate important-term list automatically
important_terms_df = word_stats[
    (word_stats["total_occurrences"] >= 100) &
    (word_stats["unique_documents"] >= 50)
].copy()
display(
    important_terms_df.head(100)
)

,word,total_occurrences,unique_documents,document_coverage_pct
1,hcp,15372,10478,69.853333
9,recent,7075,6082,40.546667
4,patients,5335,4689,31.260000
60,experience,5149,5022,33.480000
192,current,3925,3642,24.280000
...,...,...,...,...
304,paper,636,629,4.193333
136,jun,630,630,4.200000
273,administration,627,621,4.140000
357,easy,625,624,4.160000


In [41]:
important_terms = set(
    important_terms_df["word"]
)
print("Number of important terms:", len(important_terms))

print(
    sorted(important_terms)
)

Number of important terms: 424
['acceptance', 'access', 'action', 'actionable', 'additional', 'address', 'addressed', 'adherence', 'administration', 'adoption', 'aerolyn', 'affected', 'affordability', 'agreed', 'aligns', 'alleriva', 'answer', 'answered', 'appear', 'approach', 'appropriate', 'approval', 'approvals', 'apr', 'arthrelis', 'ask', 'asked', 'assigned', 'aug', 'auth', 'authorization', 'availability', 'backing', 'barrier', 'barriers', 'benefit', 'better', 'biggest', 'bottleneck', 'bounced', 'brevanta', 'brief', 'broader', 'burden', 'calling', 'came', 'cardeza', 'cautious', 'celunex', 'cheaper', 'check', 'checked', 'clarification', 'cleared', 'clearer', 'clearly', 'clears', 'clearvanta', 'clinical', 'close', 'closer', 'comfortable', 'commitment', 'committed', 'comparative', 'compared', 'complaints', 'complexity', 'concern', 'confidence', 'confident', 'confirm', 'confusion', 'considerations', 'considered', 'consistent', 'consistently', 'context', 'continuation', 'continues', 'con

In [36]:
#Remove stopwords
def remove_stopwords(text):
    words = text.split()
    filtered_words = [
        word for word in words
        if word not in stop_words
    ]
    return " ".join(filtered_words)

df_clean["text_no_stopwords"] = (
    df_clean["text_no_punct"]
    .apply(remove_stopwords)
)

In [37]:
df_clean[
    ["text_no_punct", "text_no_stopwords"]
].head(10)

,text_no_punct,text_no_stopwords
0,saw hcp regarding arthrelis patients mentioned...,saw hcp regarding arthrelis patients mentioned...
1,met on arthrelis hcp reports fewer tolerance c...,met arthrelis hcp reports fewer tolerance comp...
2,met on arthrelis the previous adherence concer...,met arthrelis previous adherence concern eased...
3,the hcp reviewed recent experience with arthre...,hcp reviewed recent experience arthrelis sched...
4,brief discussion focused on rheumora payer app...,brief discussion focused rheumora payer approv...
5,quick follow up on arthrelis office reports fe...,quick follow arthrelis office reports fewer pa...
6,arthrelis discussion use in routine practice f...,arthrelis discussion use routine practice feel...
7,saw hcp regarding arthrelis hcp remains cautio...,saw hcp regarding arthrelis hcp remains cautio...
8,rheumora discussion recent approvals have gone...,rheumora discussion recent approvals gone with...
9,saw hcp regarding renovia practice wants a sim...,saw hcp regarding renovia practice wants simpl...


In [ ]:
#Lemmatization
!pip -q install spacy
!python -m spacy download en_core_web_sm
import spacy

# Create a set of important pharmaceutical terms
protected_terms = set()

for column in ["drug_name", "brand_name", "competitor_brand"]:
    if column in df_clean.columns:
        terms = (
            df_clean[column]
            .dropna()
            .astype(str)
            .str.lower()
            .str.strip()
        )
        protected_terms.update(terms)

print("Number of protected terms:", len(protected_terms))
def lemmatize_text(text):
    doc = nlp(text)

    lemmas = []

    for token in doc:
        if not token.is_alpha:
            continue

        word = token.text.lower()

        # Keep drug/brand names unchanged
        if word in protected_terms:
            lemmas.append(word)
        else:
            lemmas.append(token.lemma_.lower())

    return " ".join(lemmas)


df_clean["clean_text"] = (
    df_clean["text_no_stopwords"]
    .apply(lemmatize_text)
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 123.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Number of protected terms: 57


In [ ]:
df_clean[
    ["text_no_stopwords", "clean_text"]
].head(10).to_string(index=False)

'                                                                                                                                                                                   text_no_stopwords                                                                                                                                                                              clean_text\n                                                                                saw hcp regarding arthrelis patients mentioned injection site reaction recent outcomes favorable nothing else urgent                                                                           see hcp regard arthrelis patient mention injection site reaction recent outcome favorable nothing else urgent\n                                               met arthrelis hcp reports fewer tolerance complaints recently several patients not following regimen consistently send patient support material owner                                       

In [ ]:
#Check 5 notes
for i in range(5):
    print(f"\n--- NOTE {i+1} ---")
    print("Original :", df_clean["crm_note"].iloc[i])
    print("Cleaned  :", df_clean["clean_text"].iloc[i])


--- NOTE 1 ---
Original : Saw HCP re Arthrelis. 2 patients mentioned injection-site reaction. recent outcomes have been favorable. Nothing else urgent.
Cleaned  : see hcp regard arthrelis patient mention injection site reaction recent outcome favorable nothing else urgent

--- NOTE 2 ---
Original : Met on Arthrelis. HCP reports fewer tolerance complaints recently. several patients are not following the regimen consistently. Send patient-support material by Mar 06; owner REP0002.
Cleaned  : meet arthrelis hcp report few tolerance complaint recently several patient not follow regiman consistently send patient support material owner

--- NOTE 3 ---
Original : Met on Arthrelis. the previous adherence concern has eased. HCP wants more confidence in expected benefit. authorization turnaround remains slow. HCP wants access / pa resource; REP0002 to f/u.
Cleaned  : meet arthrelis previous adherence concern ease hcp want confidence expect benefit authorization turnaround remain slow hcp want a

In [ ]:
#Check for empty cleaned notes
print(
    "Empty clean texts:",
    df_clean["clean_text"].str.strip().eq("").sum()
)

Empty clean texts: 0


In [ ]:
#Compare word counts
df_clean["clean_word_count"] = (
    df_clean["clean_text"].str.split().str.len()
)

print("Original average:",
      round(df_clean["word_count_raw"].mean(), 2))

print("Cleaned average:",
      round(df_clean["clean_word_count"].mean(), 2))

Original average: 25.69
Cleaned average: 17.3


In [ ]:
output_file = "CTS_CRM_15000_NLP_Preprocessed.csv"

df_clean.to_csv(output_file, index=False)

print("Saved successfully:", output_file)

Saved successfully: CTS_CRM_15000_NLP_Preprocessed.csv


In [ ]:
import os

print("File exists:", os.path.exists(output_file))
print("Rows:", len(df_clean))

File exists: True
Rows: 15000
